# PhysIR — Stage 5 SET 1 — core decomposition @ 256 (Kaggle, ONE FLOW)

One flow, zero manual selections. **Uniform 200 ep** for every arm (matched table), an
**ep150 snapshot** per arm (second matched operating point — PSNR-peak zone vs deepest
perceptual), and eval at **{ep150, best, last}** × EMA weights → per-image CSVs for the
paired Wilcoxon test.

| config | losses | tests |
|--------|--------|-------|
| abl_baseline | pixel+edge+enhance | the matched reference for every delta |
| abl_phase | +phase 0.20 | phase's standalone contribution |
| abl_retinex | +retinex 0.18 | retinex's standalone contribution |

**Regime:** crop 256, eff-batch 8 (4×2-T4 DDP), lr 2.8e-4 cosine, EMA on, seed 42.
≈ 3 arms × ~28 h ≈ **84 GPU-h** → ~8 capped sessions.

## Session workflow
1. **Accelerator: GPU T4 × 2.** Inputs: `physir-code` (⚠️ re-upload with the CURRENT
   `train.py` + `losses/` — the guard cell refuses stale code), `lolblur-dataset`.
2. Run All and let it train. **No self-termination** — training runs until every arm
   finishes or the session ends. Checkpoints are written **atomically every 2 epochs**
   (weights + optimizer + scheduler + EMA + blur-σ + best-so-far), so a kill at any
   moment loses ≤ 2 epochs and resume is seamless.
3. ⚠️ A resume point only exists if the session's output was **saved as a version** —
   an interactive session that dies unsaved keeps nothing. Save a version before the
   12 h cap (or run via Save & Run All within the cap).
4. Next session: **Add Input → Your Work → this notebook** (previous version's output),
   then Run All. The restore cell auto-copies checkpoints/logs and every arm resumes
   from its checkpoint; finished arms are skipped, eval fills in as arms finish.
5. Repeat until the leaderboard prints every row. Cross-set comparisons + Wilcoxon run
   offline on the downloaded `metrics_out/per_image/` CSVs.
6. **Week-3/4 seed replicates:** change `SEED = 42` → **1337 or 2024** (→ 3 total seeds; odd-count convention) in the config cell —
   nothing else. Runs get `_s1337`/`_s2024` suffixes and coexist with the seed-42 outputs.
   (`BUDGET_H` in the config cell re-enables a clean self-stop if ever wanted; default 0.)


## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths + stale-code guard

In [ ]:
import os, sys, glob
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
WORKING_DIR = '/kaggle/working'
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for d in range(1, depth+1):
        h = sorted(glob.glob(os.path.join(base, *(['*']*d), marker)))
        if h: return h[0][:-len(marker)].rstrip(os.sep)
    return None
CODE_ROOT = find_root(CODE_DIR, 'train.py'); assert CODE_ROOT
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)
src = open(os.path.join(CODE_ROOT,'train.py')).read()
miss = [n for n in ('restore_phys_state','class ModelEMA','best_psnr',
                    'snapshot_epochs','time_budget_h','Seeded RNGs','_atomic_save') if n not in src]
lsrc = open(os.path.join(CODE_ROOT,'losses','loss.py')).read()
if 'class LPIPSLoss' not in lsrc: miss.append('class LPIPSLoss')
assert not miss, f'STALE physir-code, missing {miss} — re-upload with the current train.py + losses/.'
LOLBLUR = find_root(LOLBLUR_ROOT, os.path.join('train','high_sharp_scaled')) or LOLBLUR_ROOT
cand = find_root(LOLBLUR_ROOT, os.path.join('test','low_blur_noise'))
LOLBLUR = cand or LOLBLUR
print('CODE_ROOT:', CODE_ROOT, '| features present ✅')
print('LOLBLUR  :', LOLBLUR, '| has test/:', os.path.isdir(os.path.join(LOLBLUR,'test')))

## 2b. Session restore (attach previous version's output as an input)

In [ ]:
# Continuation sessions: Add Input → Your Work → this notebook (previous version's
# output), then just Run All. Auto-detects checkpoints/logs in ANY attached input and
# copies them into /kaggle/working — fresh sessions print 'nothing to restore'.
import shutil
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/abl_*.pt', '/kaggle/input/*/*/abl_*.pt',
                '/kaggle/input/*/*/*/abl_*.pt'):
        for src in glob.glob(pat):
            dst = os.path.join(WORKING_DIR, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst); n += 1
    # run logs live in visible metrics_out/ (Kaggle outputs can drop dot-dirs)
    logs = []
    for base in ('metrics_out', '.metrics'):
        logs += glob.glob(f'/kaggle/input/*/{base}/*/*.csv')
        logs += glob.glob(f'/kaggle/input/*/*/{base}/*/*.csv')
        logs += glob.glob(f'/kaggle/input/*/*/*/{base}/*/*.csv')
    for src in logs:
        run  = os.path.basename(os.path.dirname(src))
        dstd = os.path.join(WORKING_DIR, '.metrics', run)
        os.makedirs(dstd, exist_ok=True)
        dst = os.path.join(dstd, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst); n += 1
    print(f'restored {n} files from previous session' if n else 'fresh session — nothing to restore')
_restore()

## 3. Train/val split (90/10, seed 42 — same as all stages)

In [ ]:
import random
from pathlib import Path
SUBS=['low_blur_noise','high_sharp_scaled']
TRAIN_SRC=Path(LOLBLUR)/'train'; SPLIT=Path(WORKING_DIR)/'LOLBlur_split'
random.seed(42)   # split seed is FIXED across all stages/seeds — never change
clips=sorted([d.name for d in (TRAIN_SRC/'low_blur_noise').iterdir() if d.is_dir()])
n_val=max(1,int(len(clips)*0.1)); val=set(random.sample(clips,n_val)); tr=[c for c in clips if c not in val]
for nm,cl in [('physir_train',tr),('physir_val',val)]:
    for sub in SUBS:
        b=SPLIT/nm/sub; b.mkdir(parents=True,exist_ok=True)
        for c in cl:
            d=b/c
            if not d.exists(): d.symlink_to((TRAIN_SRC/sub/c).resolve())
PHYSIR_TRAIN=str(SPLIT/'physir_train'); PHYSIR_VAL=str(SPLIT/'physir_val')
print('train',len(tr),'val',len(val))

## 4. Configs — uniform 200 ep, ep150 snapshot, seed knob

In [ ]:
import yaml
EPOCHS   = 200      # UNIFORM across all ablation arms (matched table — do not vary per arm)
SNAPSHOT = 150      # frozen mid-run checkpoint → second matched operating point (PSNR-peak zone)
SEED     = 42       # change ONLY for the replicate pass (1337, then 2024 → 3 total seeds); split stays 42
RUN_TAG  = '' if SEED == 42 else f'_s{SEED}'
PERCEP_W = 0.01     # DarkIR's lambda_pe (Eq.9); LPIPS(VGG19) = DarkIR's L_percep
BUDGET_H = 0        # optional clean self-stop after N hours; 0 = DISABLED — train until the
                    # session ends; atomic 2-epoch checkpoints make any kill resume-safe

# each: (edge_on, phase_w, retinex_w, blur_w, percep_on)
CONFIGS = {
  'abl_baseline': dict(edge=True,  phase=0.0,  retinex=0.0,  blur=0.0,  percep=False),
  'abl_phase':    dict(edge=True,  phase=0.20, retinex=0.0,  blur=0.0,  percep=False),
  'abl_retinex':  dict(edge=True,  phase=0.0,  retinex=0.18, blur=0.0,  percep=False),
}
CONFIGS = {k + RUN_TAG: v for k, v in CONFIGS.items()}

def build_cfg(name, sp, budget_h=0):
    phys_on = (sp['phase']>0 or sp['retinex']>0 or sp['blur']>0)
    cfg = {
      'seed': SEED,
      'datasets': {'name':'LOLBlur',
        'train': {'low_path':f'{PHYSIR_TRAIN}/low_blur_noise','high_path':f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size':256,'batch_size':4,'n_workers':4,'use_flips':True},
        'val': {'test_path':PHYSIR_VAL,'batch_size_test':1,'verbose':False,'n_workers':4}},
      'network': {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
                  'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True},
      'save': {'pretrained':None,'path':f'{WORKING_DIR}/{name}_latest.pt','best':f'{WORKING_DIR}/{name}_best.pt',
               'save_freq':2,'eval_freq':10,'perceptual_eval_epochs':[],
               'snapshot_epochs':[SNAPSHOT]},
      'pixel_criterion':'l1',
      'perceptual': sp['percep'], 'perceptual_weight':PERCEP_W,
      'perceptual_type':'lpips','perceptual_net':'vgg',   # faithful DarkIR L_percep = LPIPS(VGG19)
      'perceptual_criterion':'l1','perceptual_reduction':'mean',
      'edge': sp['edge'],'edge_weight':0.05,'edge_criterion':'l2','edge_reduction':'mean',
      'frequency':False,
      'enhance':True,'enhance_weight':0.01,'enhance_criterion':'l1','enhance_reduction':'mean',
      'physics_losses': {'enabled':phys_on,'phase2_start':9999,'phase3_start':9999,'phase4_start':1,
                         'retinex': sp['retinex']>0,'retinex_weight':sp['retinex'],
                         'blur': sp['blur']>0,'blur_weight':sp['blur'],
                         'phase': sp['phase']>0,'phase_weight':sp['phase']},
      'train': {'epochs':EPOCHS,'lr_initial':0.00028,'weight_decay':0.001,'lr_scheme':'CosineAnnealing',
                'eta_min':1e-6,'grad_clip':1.0,'time_budget_h':budget_h,
                'ema':{'enabled':True,'decay':0.9999,'warmup':2000}},
      'wandb':{'init':False,'name':name},'Resize':False,
    }
    p=f'{WORKING_DIR}/{name}.yml'
    yaml.dump(cfg, open(p,'w'), default_flow_style=False)
    return p, cfg
print('configs:', list(CONFIGS), '| EPOCHS', EPOCHS, '| snapshot @', SNAPSHOT, '| seed', SEED)

## 5. TRAIN loop — idempotent, resume-safe, session-budgeted

In [ ]:
import csv, subprocess
import torch

def done_ep(name):
    # Ground truth = the resume checkpoint itself (what train.py will actually resume from);
    # the training log is only a fallback if the checkpoint is missing/unreadable.
    ck = os.path.join(WORKING_DIR, f'{name}_latest.pt')
    if os.path.isfile(ck):
        try:
            return int(torch.load(ck, map_location='cpu', weights_only=False).get('epoch', 0))
        except Exception as e:
            print(f'  ({name}_latest.pt unreadable: {e} — falling back to log)')
    lg = Path(WORKING_DIR)/'.metrics'/name/'training_log.csv'
    if not lg.exists(): return 0
    last = 0
    try:
        for r in csv.DictReader(open(lg)): last = max(last, int(r['epoch']))
    except Exception: pass
    return last

def mirror_metrics():
    src = os.path.join(WORKING_DIR, '.metrics'); dst = os.path.join(WORKING_DIR, 'metrics_out')
    if os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)

# Runs until every arm reaches EPOCHS or Kaggle ends the session — no self-termination
# (BUDGET_H=0). Atomic 2-epoch checkpoints mean a kill at ANY moment loses ≤2 epochs and
# resumes seamlessly next session (weights, optimizer, scheduler, EMA, blur-σ, best-so-far).
for name, sp in CONFIGS.items():
    d = done_ep(name)
    if d >= EPOCHS:
        print(f'✅ {name}: trained {d}/{EPOCHS} ep — skip'); continue
    p, _ = build_cfg(name, sp, budget_h=BUDGET_H)
    tag = f'(resume ep {d})' if d > 0 else '(scratch)'
    print(f'\n===== TRAIN {name} {tag} → {EPOCHS} ep =====')
    r = subprocess.run(['torchrun','--nproc_per_node=2', f'{CODE_ROOT}/train.py', '-p', p])
    print(f'===== {name} exit {r.returncode} | now {done_ep(name)}/{EPOCHS} ep =====')
    mirror_metrics()
mirror_metrics()
print('\nTRAIN loop done —', {n: f'{done_ep(n)}/{EPOCHS}' for n in CONFIGS})

## 6. EVAL loop — LOLBlur test × {ep150, best, last} + per-image CSVs

In [ ]:
import torch, numpy as np, csv
import train as T
from data import create_test_data
device='cuda'
opt_test={'name':'LOLBlur','val':{'test_path':os.path.join(LOLBLUR,'test'),'batch_size_test':1},
          'train':{'n_workers':4,'verbose':True}}
tl_raw,_=create_test_data(rank=0,world_size=1,opt=opt_test)
tl = tl_raw if not isinstance(tl_raw,dict) else list(tl_raw.values())[0]['loader']
lpips_fn = T._LPIPS(net='vgg',verbose=False).to(device) if T._LPIPS_AVAILABLE else None
PERIMG=Path(WORKING_DIR)/'.metrics'/'per_image'; PERIMG.mkdir(parents=True,exist_ok=True)

net_cfg={'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
         'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

@torch.no_grad()
def eval_test(model, out_csv):
    rows=[]
    for i,(high,low) in enumerate(tl):
        high,low=high.to(device),low.to(device)
        pred=model(low,side_loss=False).clamp(0,1)
        _,_,H,W=high.shape; pred=pred[:,:,:H,:W]
        mse=torch.mean((pred-high)**2).item()
        psnr=20*np.log10(1.0/(np.sqrt(mse)+1e-8))
        T._ssim_metric.to(device); ssim=T._ssim_metric(pred,high).item()
        lp=lpips_fn(pred*2-1,high*2-1).mean().item()
        rows.append((i,psnr,ssim,lp))
    with open(out_csv,'w',newline='') as f:
        w=csv.writer(f); w.writerow(['idx','psnr','ssim','lpips']); w.writerows(rows)
    a=np.array([(p,s,l) for _,p,s,l in rows])
    return {'psnr':a[:,0].mean(),'ssim':a[:,1].mean(),'lpips':a[:,2].mean(),'n':len(a)}

# three operating points per arm: ep150 snapshot (PSNR-peak zone), best (PSNR-selected),
# last (ep200, deepest perceptual). All EMA weights.
def ckpts(name):
    return [('ep150', f'{WORKING_DIR}/{name}_latest_ep{SNAPSHOT}.pt'),
            ('best',  f'{WORKING_DIR}/{name}_best.pt'),
            ('last',  f'{WORKING_DIR}/{name}_latest.pt')]

for name in CONFIGS:
    if done_ep(name) < EPOCHS:
        print(f'⏳ {name}: {done_ep(name)}/{EPOCHS} ep — finish training first'); continue
    for tag, ck_path in ckpts(name):
        csvp = PERIMG/f'per_image_{name}_{tag}.csv'
        if csvp.exists():
            print(f'✅ {name}[{tag}]: per-image exists — skip'); continue
        if not os.path.isfile(ck_path):
            print(f'⚠️ {name}[{tag}]: {os.path.basename(ck_path)} missing'); continue
        ck=torch.load(ck_path,map_location='cpu',weights_only=False)
        model=T.build_model(net_cfg).to(device)
        state=ck.get('ema') or ck['model']
        model.load_state_dict({k.replace('module.',''):v for k,v in state.items()},strict=False); model.eval()
        m=eval_test(model,csvp)
        ep_at = ck.get('best_epoch') if tag=='best' else ck.get('epoch')
        print(f'{name:22s}[{tag:5s}@ep{ep_at}] PSNR={m["psnr"]:.4f} SSIM={m["ssim"]:.4f} '
              f'LPIPS={m["lpips"]:.4f}  -> {csvp.name}')
        del model; torch.cuda.empty_cache()
mirror_metrics()
print('\nEVAL loop complete.')

## 7. Leaderboard

In [ ]:
import pandas as pd
rows=[]
for name in CONFIGS:
    for tag in ('ep150','best','last'):
        csvp=Path(WORKING_DIR)/'.metrics'/'per_image'/f'per_image_{name}_{tag}.csv'
        if not csvp.exists(): continue
        df=pd.read_csv(csvp)
        rows.append({'config':name,'ckpt':tag,'psnr':df['psnr'].mean(),'ssim':df['ssim'].mean(),
                     'lpips':df['lpips'].mean(),'n':len(df)})
if rows:
    b=pd.DataFrame(rows)
    print('=== SET 1 — core decomposition @256/200 — LOLBlur test ===')
    print(b.to_string(index=False))
    for metric in ('psnr','lpips'):
        piv=b.pivot(index='config',columns='ckpt',values=metric)
        print(f'\n{metric.upper()} by operating point:'); print(piv.round(4).to_string())
    ref=f'abl_baseline{RUN_TAG}'
    if ref in set(b['config']):
        base=b[b['config']==ref]
        print('\ndeltas vs', ref, '(same operating point):')
        for _,r in b.iterrows():
            m=base[base['ckpt']==r['ckpt']]
            if len(m): print(f"  {r['config']:22s}[{r['ckpt']:5s}] {r['psnr']-m.iloc[0]['psnr']:+.4f} dB")
    print('\nper-image CSVs in metrics_out/per_image/ → download; cross-SET deltas + '
          'paired_wilcoxon.py run offline against the other sets.')
else:
    print('no results yet — run train + eval loops')